Project Overview :
In this notebook, we performed fine-tuning of a pre-trained BERT Generation Encoder-Decoder model,
which combines a BertGenerationEncoder and a BertGenerationDecoder, to generate new text in Shakespearean style.
We used a ready-made model and libraries (transformers), so there was no need to train a model from scratch — we only adapted it to our data.
For fine-tuning, we downloaded The Complete Works of Shakespeare (pg100.txt) from Project Gutenberg:
https://www.gutenberg.org/cache/epub/100/pg100.txt

We created input-output pairs of 50-word sequences as input and the same sequence shifted by one word as output.

We trained and saved models under several configurations for comparison:

30,000 pairs for 1 epoch (30K EPOCH 1)

30,000 pairs for 2 epochs (30K EPOCH 2)

100,000 pairs for 1 epoch (100K EPOCH 1)

100,000 pairs for 2 epochs (100K EPOCH 2)

100,000 pairs for 3 epochs (100K EPOCH 3)

 These trained models were first saved in our Google Drive for future use and performance comparison.
Later, we also uploaded them to Hugging Face and made them public so they are accessible for sharing, evaluation, and reuse.
You can view and access all of these models and the evaluation dataset at:
https://huggingface.co/NahlaAboromi

Finally, we built an interactive interface that uses these models directly from Hugging Face for generating Shakespearean-style text based on user input.



In [ ]:

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install transformers==4.28.1


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 110.0/110.0 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.0/7.0 MB 61.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 82.7 MB/s eta 0:00:00
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.21.1
    Uninstalling tokenizers-0.21.1:
      Successfully uninstalled tokenizers-0.21.1
  Attempting uninstall: transformers
    Found existing installation: transformers 4.52.3
    Uninstalling transformers-4.52.3:
      Successfully uninstalled transformers-4.52.3
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sentence-transformers 4.1.0 requires transformers<5.0.0,>=4.41.0, but you have transformers 4.28.1 which is incompatible.


In [ ]:
# 🚀 התקנת ספריות
!pip install transformers datasets

# 📚 ייבוא ספריות
import requests
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import (
    BertTokenizer,
    BertGenerationEncoder,
    BertGenerationDecoder,
    EncoderDecoderModel,
)
from torch.optim import AdamW
import os

# ⚡ בדיקת חומרה
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('✅ Using device:', device)

# 📥 הורדת טקסט של שייקספיר
print('Downloading Shakespeare text...')
url = 'https://www.gutenberg.org/cache/epub/100/pg100.txt'
shakespeare_text = requests.get(url).text

# ✂ יצירת זוגות קלט-פלט
def create_pairs(text, block_size=50, max_pairs=100000):
    words = text.split()
    pairs = []
    for i in range(len(words) - block_size - 1):
        input_text = ' '.join(words[i:i + block_size])
        output_text = ' '.join(words[i + 1:i + block_size + 1])
        pairs.append({'input': input_text, 'output': output_text})
        if len(pairs) >= max_pairs:
            break
    return pairs

# 📚 טעינת טוקניזר ומודלים
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
tokenizer.bos_token_id = 101
tokenizer.eos_token_id = 102
tokenizer.pad_token_id = 0

encoder = BertGenerationEncoder.from_pretrained('bert-base-uncased', bos_token_id=101, eos_token_id=102)
decoder = BertGenerationDecoder.from_pretrained(
    'bert-base-uncased',
    add_cross_attention=True,
    is_decoder=True,
    bos_token_id=101,
    eos_token_id=102
)

model = EncoderDecoderModel(encoder=encoder, decoder=decoder)
model.config.decoder_start_token_id = tokenizer.bos_token_id
model.decoder.config.decoder_start_token_id = tokenizer.bos_token_id
model.config.pad_token_id = tokenizer.pad_token_id
model.config.eos_token_id = tokenizer.eos_token_id
model.to(device)

# 📦 בניית Dataset
class ShakespeareDataset(Dataset):
    def __init__(self, pairs, tokenizer, max_length=128):
        self.pairs = pairs
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        item = self.pairs[idx]
        input_enc = self.tokenizer(item['input'], padding='max_length', truncation=True,
                                   max_length=self.max_length, return_tensors='pt')
        output_enc = self.tokenizer(item['output'], padding='max_length', truncation=True,
                                    max_length=self.max_length, return_tensors='pt')
        return {
            'input_ids': input_enc.input_ids.squeeze(),
            'attention_mask': input_enc.attention_mask.squeeze(),
            'labels': output_enc.input_ids.squeeze()
        }

✅ Using device: cuda


/usr/local/lib/python3.11/dist-packages/huggingface_hub/file_download.py:943: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

You are using a model of type bert to instantiate a model of type bert-generation. This is not supported for all configurations of models and can yield errors.


model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of the model checkpoint at bert-base-uncased were not used when initializing BertGenerationEncoder: ['cls.predictions.transform.dense.weight', 'cls.seq_relationship.bias', 'bert.pooler.dense.bias', 'bert.embeddings.token_type_embeddings.weight', 'cls.predictions.bias', 'cls.predictions.transform.LayerNorm.weight', 'cls.predictions.transform.dense.bias', 'cls.predictions.transform.LayerNorm.bias', 'bert.pooler.dense.weight', 'cls.seq_relationship.weight']
- This IS expected if you are initializing BertGenerationEncoder from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertGenerationEncoder from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
You are using a model of type bert to instantiate a

In [ ]:
# 📊 אימון עבור 30K עם 1 אפוקים
pairs = create_pairs(shakespeare_text, max_pairs=30000)
dataset = ShakespeareDataset(pairs, tokenizer)
dataloader = DataLoader(dataset, batch_size=16, shuffle=True)

optimizer = AdamW(model.parameters(), lr=5e-5)
model.train()
print('🔥 Starting training...')
import time
start_time = time.time()

for epoch in range(1):
    total_loss = 0
    for i, batch in enumerate(dataloader):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )
        loss = outputs.loss
        total_loss += loss.item()
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        if (i + 1) % 100 == 0:
            print(f'Epoch {epoch+1} Batch {i+1} Loss: {loss.item():.4f}')
    print(f'✅ Epoch {epoch+1} Total Loss: {total_loss:.4f}')

# 💾 שמירת המודל
output_dir = "/content/drive/MyDrive/shakespeare_bert_base_30KEPOCH1"
os.makedirs(output_dir, exist_ok=True)
model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)
print(f"📁 Model saved to: {output_dir}")

!zip -r shakespeare_bert_base_30KEPOCH1.zip /content/drive/MyDrive/shakespeare_bert_base_30KEPOCH1
from google.colab import files
files.download("shakespeare_bert_base_30KEPOCH1.zip")
end_time = time.time()
print(f"⏱ Total training time: {(end_time - start_time)/60:.2f} minutes")

🔥 Starting training...


/usr/local/lib/python3.11/dist-packages/transformers/models/encoder_decoder/modeling_encoder_decoder.py:634: FutureWarning: Version v4.12.0 introduces a better way to train encoder-decoder models by computing the loss inside the encoder-decoder framework rather than in the decoder itself. You may observe training discrepancies if fine-tuning a model trained with versions anterior to 4.12.0. The decoder_input_ids are now created based on the labels, no need to pass them yourself anymore.
  warnings.warn(DEPRECATION_WARNING, FutureWarning)


Epoch 1 Batch 100 Loss: 3.2982
Epoch 1 Batch 200 Loss: 1.2608
Epoch 1 Batch 300 Loss: 0.8422
Epoch 1 Batch 400 Loss: 0.6177
Epoch 1 Batch 500 Loss: 0.3848
Epoch 1 Batch 600 Loss: 0.3448
Epoch 1 Batch 700 Loss: 0.1890
Epoch 1 Batch 800 Loss: 0.1715
Epoch 1 Batch 900 Loss: 0.1268
Epoch 1 Batch 1000 Loss: 0.1151
Epoch 1 Batch 1100 Loss: 0.0805
Epoch 1 Batch 1200 Loss: 0.0768
Epoch 1 Batch 1300 Loss: 0.0594
Epoch 1 Batch 1400 Loss: 0.0502
Epoch 1 Batch 1500 Loss: 0.0417
Epoch 1 Batch 1600 Loss: 0.0362
Epoch 1 Batch 1700 Loss: 0.0318
Epoch 1 Batch 1800 Loss: 0.0317
✅ Epoch 1 Total Loss: 1112.7310
📁 Model saved to: /content/drive/MyDrive/shakespeare_bert_base_30KEPOCH1
  adding: content/drive/MyDrive/shakespeare_bert_base_30KEPOCH1/ (stored 0%)
  adding: content/drive/MyDrive/shakespeare_bert_base_30KEPOCH1/config.json (deflated 81%)
  adding: content/drive/MyDrive/shakespeare_bert_base_30KEPOCH1/generation_config.json (deflated 29%)
  adding: content/drive/MyDrive/shakespeare_bert_base_30KE

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

⏱ Total training time: 32.61 minutes


In [ ]:
# 📊 אימון עבור 30K עם 2 אפוקים
pairs = create_pairs(shakespeare_text, max_pairs=30000)
dataset = ShakespeareDataset(pairs, tokenizer)
dataloader = DataLoader(dataset, batch_size=16, shuffle=True)

optimizer = AdamW(model.parameters(), lr=5e-5)
model.train()
print('🔥 Starting training...')
import time
start_time = time.time()

for epoch in range(2):
    total_loss = 0
    for i, batch in enumerate(dataloader):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )
        loss = outputs.loss
        total_loss += loss.item()
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        if (i + 1) % 100 == 0:
            print(f'Epoch {epoch+1} Batch {i+1} Loss: {loss.item():.4f}')
    print(f'✅ Epoch {epoch+1} Total Loss: {total_loss:.4f}')


# 💾 שמירת המודל
output_dir = "/content/drive/MyDrive/shakespeare_bert_base_30KEPOCH2"
os.makedirs(output_dir, exist_ok=True)
model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)
print(f"📁 Model saved to: {output_dir}")

!zip -r shakespeare_bert_base_30KEPOCH2.zip /content/drive/MyDrive/shakespeare_bert_base_30KEPOCH2
from google.colab import files
files.download("shakespeare_bert_base_30KEPOCH2.zip")
end_time = time.time()
print(f"⏱ Total training time: {(end_time - start_time)/60:.2f} minutes")

🔥 Starting training...
Epoch 1 Batch 100 Loss: 0.0165
Epoch 1 Batch 200 Loss: 0.0160
Epoch 1 Batch 300 Loss: 0.0080
Epoch 1 Batch 400 Loss: 0.0087
Epoch 1 Batch 500 Loss: 0.0093
Epoch 1 Batch 600 Loss: 0.0045
Epoch 1 Batch 700 Loss: 0.0031
Epoch 1 Batch 800 Loss: 0.0030
Epoch 1 Batch 900 Loss: 0.0034
Epoch 1 Batch 1000 Loss: 0.0023
Epoch 1 Batch 1100 Loss: 0.0023
Epoch 1 Batch 1200 Loss: 0.0019
Epoch 1 Batch 1300 Loss: 0.0023
Epoch 1 Batch 1400 Loss: 0.0023
Epoch 1 Batch 1500 Loss: 0.0018
Epoch 1 Batch 1600 Loss: 0.0016
Epoch 1 Batch 1700 Loss: 0.0024
Epoch 1 Batch 1800 Loss: 0.0013
✅ Epoch 1 Total Loss: 10.5189
Epoch 2 Batch 100 Loss: 0.0013
Epoch 2 Batch 200 Loss: 0.0038
Epoch 2 Batch 300 Loss: 0.0010
Epoch 2 Batch 400 Loss: 0.0044
Epoch 2 Batch 500 Loss: 0.0009
Epoch 2 Batch 600 Loss: 0.0049
Epoch 2 Batch 700 Loss: 0.0025
Epoch 2 Batch 800 Loss: 0.0013
Epoch 2 Batch 900 Loss: 0.0013
Epoch 2 Batch 1000 Loss: 0.0009
Epoch 2 Batch 1100 Loss: 0.0016
Epoch 2 Batch 1200 Loss: 0.0009
Epoch

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

⏱ Total training time: 64.27 minutes


In [ ]:
# 📊 אימון עבור 100K עם 1 אפוקים
pairs = create_pairs(shakespeare_text, max_pairs=100000)
dataset = ShakespeareDataset(pairs, tokenizer)
dataloader = DataLoader(dataset, batch_size=16, shuffle=True)

optimizer = AdamW(model.parameters(), lr=5e-5)
model.train()
print('🔥 Starting training...')
import time
start_time = time.time()

for epoch in range(1):
    total_loss = 0
    for i, batch in enumerate(dataloader):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )
        loss = outputs.loss
        total_loss += loss.item()
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        if (i + 1) % 100 == 0:
            print(f'Epoch {epoch+1} Batch {i+1} Loss: {loss.item():.4f}')
    print(f'✅ Epoch {epoch+1} Total Loss: {total_loss:.4f}')

# 💾 שמירת המודל
output_dir = "/content/drive/MyDrive/shakespeare_bert_base_100KEPOCH1"
os.makedirs(output_dir, exist_ok=True)
model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)
print(f"📁 Model saved to: {output_dir}")

!zip -r shakespeare_bert_base_100KEPOCH1.zip /content/drive/MyDrive/shakespeare_bert_base_100KEPOCH1
from google.colab import files
files.download("shakespeare_bert_base_100KEPOCH1.zip")
end_time = time.time()
print(f"⏱ Total training time: {(end_time - start_time)/60:.2f} minutes")

🔥 Starting training...


/usr/local/lib/python3.11/dist-packages/transformers/models/encoder_decoder/modeling_encoder_decoder.py:634: FutureWarning: Version v4.12.0 introduces a better way to train encoder-decoder models by computing the loss inside the encoder-decoder framework rather than in the decoder itself. You may observe training discrepancies if fine-tuning a model trained with versions anterior to 4.12.0. The decoder_input_ids are now created based on the labels, no need to pass them yourself anymore.
  warnings.warn(DEPRECATION_WARNING, FutureWarning)


Epoch 1 Batch 100 Loss: 2.8546
Epoch 1 Batch 200 Loss: 1.2323
Epoch 1 Batch 300 Loss: 0.7419
Epoch 1 Batch 400 Loss: 0.6197
Epoch 1 Batch 500 Loss: 0.4282
Epoch 1 Batch 600 Loss: 0.2903
Epoch 1 Batch 700 Loss: 0.3236
Epoch 1 Batch 800 Loss: 0.2653
Epoch 1 Batch 900 Loss: 0.1876
Epoch 1 Batch 1000 Loss: 0.1804
Epoch 1 Batch 1100 Loss: 0.1495
Epoch 1 Batch 1200 Loss: 0.1559
Epoch 1 Batch 1300 Loss: 0.0964
Epoch 1 Batch 1400 Loss: 0.1053
Epoch 1 Batch 1500 Loss: 0.0596
Epoch 1 Batch 1600 Loss: 0.0589
Epoch 1 Batch 1700 Loss: 0.0590
Epoch 1 Batch 1800 Loss: 0.0669
Epoch 1 Batch 1900 Loss: 0.0414
Epoch 1 Batch 2000 Loss: 0.0494
Epoch 1 Batch 2100 Loss: 0.0356
Epoch 1 Batch 2200 Loss: 0.0441
Epoch 1 Batch 2300 Loss: 0.0319
Epoch 1 Batch 2400 Loss: 0.0378
Epoch 1 Batch 2500 Loss: 0.0259
Epoch 1 Batch 2600 Loss: 0.0250
Epoch 1 Batch 2700 Loss: 0.0257
Epoch 1 Batch 2800 Loss: 0.0145
Epoch 1 Batch 2900 Loss: 0.0151
Epoch 1 Batch 3000 Loss: 0.0203
Epoch 1 Batch 3100 Loss: 0.0148
Epoch 1 Batch 320

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

⏱ Total training time: 95.68 minutes


In [ ]:
# 🚀 This script continues training a BERT EncoderDecoder model on Shakespeare text for one more epoch (Epoch 2),
# loading the previously saved model from Epoch 1 and printing ETA every 100 batches.

import os
import time
from datetime import timedelta
from transformers import BertTokenizer, EncoderDecoderModel
from torch.optim import AdamW

from torch.utils.data import DataLoader
import torch  # אל תשכחי לייבא

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 📥 Load the model saved after Epoch 1
load_path = "/content/drive/MyDrive/shakespeare_bert_base_100KEPOCH1"
model = EncoderDecoderModel.from_pretrained(load_path).to(device)
tokenizer = BertTokenizer.from_pretrained(load_path)

# 🔁 Recreate the dataset (same text and pairs)
pairs = create_pairs(shakespeare_text, max_pairs=100000)
dataset = ShakespeareDataset(pairs, tokenizer)
dataloader = DataLoader(dataset, batch_size=16, shuffle=True)

# 🧠 Define optimizer
optimizer = AdamW(model.parameters(), lr=5e-5)
model.train()

# ⚙️ Continue training for 1 more epoch (Epoch 2)
total_epochs = 1
output_name = "100K_EPOCH2"
total_batches = len(dataloader)

print(f"🔥 Continuing training for {total_epochs} more epoch(s)...")
start_time = time.time()

for epoch in range(total_epochs):
    epoch_start = time.time()
    total_loss = 0

    for i, batch in enumerate(dataloader):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )
        loss = outputs.loss
        total_loss += loss.item()
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        # ETA calculation
        batches_done = epoch * total_batches + (i + 1)
        total_done = total_epochs * total_batches
        elapsed = time.time() - start_time
        avg_time = elapsed / batches_done
        remaining = avg_time * (total_done - batches_done)
        eta = str(timedelta(seconds=int(remaining)))

        if (i + 1) % 100 == 0 or i == total_batches - 1:
            print(f"Epoch {epoch+1} Batch {i+1}/{total_batches} Loss: {loss.item():.4f} | ETA: {eta}")

    print(f'✅ Epoch {epoch+1} complete. Total Loss: {total_loss:.4f} | Duration: {timedelta(seconds=int(time.time() - epoch_start))}')

# 💾 Save the updated model to a new directory for Epoch 2
output_dir = f"/content/drive/MyDrive/shakespeare_bert_base_{output_name}"
os.makedirs(output_dir, exist_ok=True)
model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)

print(f"📁 Model saved to: {output_dir}")


🔥 Continuing training for 1 more epoch(s)...


/usr/local/lib/python3.11/dist-packages/transformers/models/encoder_decoder/modeling_encoder_decoder.py:557: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than tensor.new_tensor(sourceTensor).
  decoder_attention_mask = decoder_input_ids.new_tensor(decoder_input_ids != self.config.pad_token_id)
/usr/local/lib/python3.11/dist-packages/transformers/models/encoder_decoder/modeling_encoder_decoder.py:577: FutureWarning: Version v4.12.0 introduces a better way to train encoder-decoder models by computing the loss inside the encoder-decoder framework rather than in the decoder itself. You may observe training discrepancies if fine-tuning a model trained with versions anterior to 4.12.0. The decoder_input_ids are now created based on the labels, no need to pass them yourself anymore.
  warnings.warn(DEPRECATION_WARNING, FutureWarning)


Epoch 1 Batch 100/6250 Loss: 0.0477 | ETA: 1:33:25
Epoch 1 Batch 200/6250 Loss: 0.0459 | ETA: 1:32:01
Epoch 1 Batch 300/6250 Loss: 0.0516 | ETA: 1:30:24
Epoch 1 Batch 400/6250 Loss: 0.0579 | ETA: 1:28:45
Epoch 1 Batch 500/6250 Loss: 0.0509 | ETA: 1:27:11
Epoch 1 Batch 600/6250 Loss: 0.0560 | ETA: 1:25:38
Epoch 1 Batch 700/6250 Loss: 0.0654 | ETA: 1:24:05
Epoch 1 Batch 800/6250 Loss: 0.0603 | ETA: 1:22:33
Epoch 1 Batch 900/6250 Loss: 0.0541 | ETA: 1:21:01
Epoch 1 Batch 1000/6250 Loss: 0.0462 | ETA: 1:19:30
Epoch 1 Batch 1100/6250 Loss: 0.0598 | ETA: 1:17:58
Epoch 1 Batch 1200/6250 Loss: 0.0698 | ETA: 1:16:27
Epoch 1 Batch 1300/6250 Loss: 0.0525 | ETA: 1:14:56
Epoch 1 Batch 1400/6250 Loss: 0.0609 | ETA: 1:13:24
Epoch 1 Batch 1500/6250 Loss: 0.0567 | ETA: 1:11:53
Epoch 1 Batch 1600/6250 Loss: 0.0627 | ETA: 1:10:22
Epoch 1 Batch 1700/6250 Loss: 0.0464 | ETA: 1:08:51
Epoch 1 Batch 1800/6250 Loss: 0.0544 | ETA: 1:07:20
Epoch 1 Batch 1900/6250 Loss: 0.0724 | ETA: 1:05:49
Epoch 1 Batch 2000/62

In [ ]:
# 🚀 This script continues training a BERT EncoderDecoder model on Shakespeare text for one more epoch (Epoch 3),
# loading the previously saved model from Epoch 2 and printing ETA every 100 batches.

import os
import time
from datetime import timedelta
from transformers import BertTokenizer, EncoderDecoderModel, AdamW
from torch.utils.data import DataLoader

# 📥 טען את המודל השמור מה-EPOCH2
load_path = "/content/drive/MyDrive/shakespeare_bert_base_100K_EPOCH2"
model = EncoderDecoderModel.from_pretrained(load_path).to(device)
tokenizer = BertTokenizer.from_pretrained(load_path)

# 🔁 צור את הנתונים שוב (אותו טקסט, אותם זוגות)
pairs = create_pairs(shakespeare_text, max_pairs=100000)
dataset = ShakespeareDataset(pairs, tokenizer)
dataloader = DataLoader(dataset, batch_size=16, shuffle=True)

# 🧠 הגדר אופטימייזר
optimizer = AdamW(model.parameters(), lr=5e-5)
model.train()

# ⚙️ המשך אימון לעוד 1 אפוק (כלומר Epoch 3)
total_epochs = 1
output_name = "100K_EPOCH3"
total_batches = len(dataloader)

print(f"🔥 Continuing training for {total_epochs} more epoch(s)...")
start_time = time.time()

for epoch in range(total_epochs):
    epoch_start = time.time()
    total_loss = 0

    for i, batch in enumerate(dataloader):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )
        loss = outputs.loss
        total_loss += loss.item()
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        # ETA חישוב
        batches_done = epoch * total_batches + (i + 1)
        total_done = total_epochs * total_batches
        elapsed = time.time() - start_time
        avg_time = elapsed / batches_done
        remaining = avg_time * (total_done - batches_done)
        eta = str(timedelta(seconds=int(remaining)))

        if (i + 1) % 100 == 0 or i == total_batches - 1:
            print(f"Epoch {epoch+1} Batch {i+1}/{total_batches} Loss: {loss.item():.4f} | ETA: {eta}")

    print(f'✅ Epoch {epoch+1} complete. Total Loss: {total_loss:.4f} | Duration: {timedelta(seconds=int(time.time() - epoch_start))}')

# 💾 שמירה לתיקייה נפרדת עבור EPOCH3
output_dir = f"/content/drive/MyDrive/shakespeare_bert_base_{output_name}"
os.makedirs(output_dir, exist_ok=True)
model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)

print(f"📁 Model saved to: {output_dir}")


Some weights of EncoderDecoderModel were not initialized from the model checkpoint at /content/drive/MyDrive/shakespeare_bert_base_100K_EPOCH2 and are newly initialized: ['encoder.embeddings.position_ids', 'decoder.bert.embeddings.position_ids']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/usr/local/lib/python3.11/dist-packages/transformers/optimization.py:391: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(


🔥 Continuing training for 1 more epoch(s)...


/usr/local/lib/python3.11/dist-packages/transformers/models/encoder_decoder/modeling_encoder_decoder.py:634: FutureWarning: Version v4.12.0 introduces a better way to train encoder-decoder models by computing the loss inside the encoder-decoder framework rather than in the decoder itself. You may observe training discrepancies if fine-tuning a model trained with versions anterior to 4.12.0. The decoder_input_ids are now created based on the labels, no need to pass them yourself anymore.
  warnings.warn(DEPRECATION_WARNING, FutureWarning)


Epoch 1 Batch 100/6250 Loss: 0.2440 | ETA: 1:29:51
Epoch 1 Batch 200/6250 Loss: 0.1676 | ETA: 1:30:10
Epoch 1 Batch 300/6250 Loss: 0.1401 | ETA: 1:29:29
Epoch 1 Batch 400/6250 Loss: 0.1081 | ETA: 1:28:22
Epoch 1 Batch 500/6250 Loss: 0.0805 | ETA: 1:27:07
Epoch 1 Batch 600/6250 Loss: 0.0790 | ETA: 1:25:46
Epoch 1 Batch 700/6250 Loss: 0.0809 | ETA: 1:24:21
Epoch 1 Batch 800/6250 Loss: 0.0889 | ETA: 1:22:54
Epoch 1 Batch 900/6250 Loss: 0.0598 | ETA: 1:21:27
Epoch 1 Batch 1000/6250 Loss: 0.0561 | ETA: 1:19:58
Epoch 1 Batch 1100/6250 Loss: 0.0509 | ETA: 1:18:28
Epoch 1 Batch 1200/6250 Loss: 0.0382 | ETA: 1:16:59
Epoch 1 Batch 1300/6250 Loss: 0.0412 | ETA: 1:15:29
Epoch 1 Batch 1400/6250 Loss: 0.0382 | ETA: 1:13:59
Epoch 1 Batch 1500/6250 Loss: 0.0458 | ETA: 1:12:29
Epoch 1 Batch 1600/6250 Loss: 0.0268 | ETA: 1:10:58
Epoch 1 Batch 1700/6250 Loss: 0.0327 | ETA: 1:09:27
Epoch 1 Batch 1800/6250 Loss: 0.0162 | ETA: 1:07:56
Epoch 1 Batch 1900/6250 Loss: 0.0174 | ETA: 1:06:26
Epoch 1 Batch 2000/62

 Interactive Interface for Shakespeare Text Generation
This interface allows users to generate Shakespearean-style text using fine-tuned BERT models.
The models are loaded directly from Hugging Face (NahlaAboromi profile), where they were uploaded after training.
Users can select between different training configurations (30k / 100k pairs, different epochs), provide a prompt,
and generate text based on the selected model.



In [ ]:
from transformers import EncoderDecoderModel, BertTokenizer
import torch
import ipywidgets as widgets
from IPython.display import display

model_paths = {
    ("30k", "EPOCH1"): "NahlaAboromi/shakespeare-bert-30k-epoch1",
    ("30k", "EPOCH2"): "NahlaAboromi/shakespeare-bert-30k-epoch2",
    ("30k", "EPOCH3"): "NahlaAboromi/shakespeare-bert-30k-epoch3",
    ("100K", "EPOCH1"): "NahlaAboromi/shakespeare-bert-100k-epoch1",
    ("100K", "EPOCH2"): "NahlaAboromi/shakespeare-bert-100k-epoch2",
    ("100K", "EPOCH3"): "NahlaAboromi/shakespeare-bert-100k-epoch3",
}


# ווידג'טים
k_selector = widgets.Dropdown(options=["30k", "100K"], description="Dataset:")
epoch_selector = widgets.Dropdown(options=["EPOCH1", "EPOCH2", "EPOCH3"], description="Epoch:")
input_text = widgets.Textarea(
    value="When I consider everything that grows",
    description="Prompt:",
    layout=widgets.Layout(width="100%", height="100px")
)
generate_button = widgets.Button(description="Generate Shakespearean Text", button_style="success")
output_area = widgets.Output()

# פעולה
def generate_text(b):
    output_area.clear_output()
    with output_area:
        selected_key = (k_selector.value, epoch_selector.value)
        if selected_key not in model_paths:
            print("⚠️ No model found for this selection.")
            return

        folder = model_paths[selected_key]

        try:
            model = EncoderDecoderModel.from_pretrained(folder)
            tokenizer = BertTokenizer.from_pretrained(folder)
            model.eval()

            inputs = tokenizer(input_text.value, return_tensors="pt")
            output = model.generate(
                input_ids=inputs.input_ids,
                max_length=50,
                num_beams=5,
                no_repeat_ngram_size=2,
                early_stopping=True
            )

            generated_text = tokenizer.decode(output[0], skip_special_tokens=True)
            print("📝 Generated Shakespearean Text:\n")
            print(generated_text)

        except Exception as e:
            print("❌ Error:", e)

generate_button.on_click(generate_text)

# הצגה
display(k_selector, epoch_selector, input_text, generate_button, output_area)


Dropdown(description='Dataset:', options=('30k', '100K'), value='30k')

Dropdown(description='Epoch:', options=('EPOCH1', 'EPOCH2', 'EPOCH3'), value='EPOCH1')

Textarea(value='When I consider everything that grows', description='Prompt:', layout=Layout(height='100px', w…

Button(button_style='success', description='Generate Shakespearean Text', style=ButtonStyle())

Output()